# Vaelo — Deal Feasibility Report Generation Pipeline

Pipeline 2 of 3 (Deal Feasibility). Same design principle as the Valuation pipeline —
**deterministic and formula-driven, no LLM, no trained model.** Every figure in the
output must trace back to a formula the CA can defend. This pipeline sits *on top of*
Pipeline 1: it takes an acquirer, a target, and proposed deal terms, and answers one
question — **is this deal financially feasible as structured?**

**Pipeline stages:**
1. Required documents & data
2. Structured intake schema
3. Deal economics — premium & sources/uses
4. Combined-entity impact — pro-forma, synergies, accretion/dilution, leverage
5. Orchestration — run the full engine
6. Report generator (templated)
7. End-to-end example run

## 1. Required Documents & Data

| Item | Why it's needed |
|---|---|
| Acquirer's latest financials (Revenue, EBITDA, Net Income, shares outstanding, net debt) | Baseline for pro-forma & accretion/dilution |
| Target's latest financials (same fields) | Baseline for pro-forma & premium check |
| Target's standalone value (from Pipeline 1, if available) | Used for premium analysis — falls back to an EV/EBITDA multiple if not supplied |
| Proposed deal terms (purchase price, cash/stock split, acquirer share price) | Defines the deal structure being tested |
| Financing plan (new debt, cost of debt, acquirer cash used, tax rate) | Determines leverage impact & interest cost |
| Synergy assumptions (cost + revenue synergies, ramp-up period, discount rate) | Determines whether the deal pays for itself |

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional
from datetime import date

## 2. Structured Intake Schema

`DealFeasibilityRequest` is the single object a CA submits for a feasibility check —
mirrors the intake boundary from Pipeline 1. `target.standalone_value` can be filled
in from a Pipeline 1 DCF run for precision; if left blank, the engine falls back to
`target.fallback_ev_ebitda_multiple` for a quick estimate.

In [ ]:
@dataclass
class DealMeta:
    """Who this report is for and why."""
    acquirer_name: str
    target_name: str
    ca_firm_name: str
    deal_rationale: str          # e.g. 'market expansion', 'vertical integration', 'succession sale'
    report_date: str = field(default_factory=lambda: date.today().isoformat())
    sector: str = "General"


@dataclass
class CompanyProfile:
    """Standalone financials for either the acquirer or the target, in Rs Cr
    unless noted. `standalone_value` and `fallback_ev_ebitda_multiple` are only
    used for the target (premium analysis)."""
    name: str
    revenue: float
    ebitda: float
    net_income: float
    shares_outstanding: float          # absolute share count
    net_debt: float                    # total debt - cash & equivalents
    standalone_value: Optional[float] = None          # from Pipeline 1 DCF, if available
    fallback_ev_ebitda_multiple: float = 6.0           # used only if standalone_value is None


@dataclass
class DealTerms:
    """The proposed deal being tested for feasibility."""
    deal_type: str                     # 'acquisition' or 'merger'
    purchase_price: float              # Rs Cr, agreed equity value for target
    cash_component_pct: float          # 0-1
    stock_component_pct: float         # 0-1, cash_component_pct + stock_component_pct = 1
    acquirer_share_price: float        # Rs, needed to size stock issuance


@dataclass
class FinancingAssumptions:
    """How the cash portion of the deal gets funded."""
    new_debt_raised: float             # Rs Cr
    cost_of_new_debt: float            # interest rate on the new debt
    acquirer_cash_used: float          # Rs Cr drawn from acquirer's own reserves
    tax_rate: float


@dataclass
class SynergyAssumptions:
    """Run-rate synergy assumptions and how fast they ramp in."""
    annual_cost_synergies: float           # Rs Cr/year at full run-rate
    annual_revenue_synergies: float        # Rs Cr/year at full run-rate
    synergy_ebitda_margin: float           # margin applied to revenue synergies for EBITDA impact
    ramp_up_years: int                     # years to reach full run-rate, linear ramp
    synergy_discount_rate: float           # for NPV — typically the combined entity's WACC


@dataclass
class DealFeasibilityRequest:
    """The single object a CA submits — the intake boundary for this pipeline."""
    meta: DealMeta
    acquirer: CompanyProfile
    target: CompanyProfile
    deal_terms: DealTerms
    financing: FinancingAssumptions
    synergies: SynergyAssumptions

## 3. Deal Economics — Premium & Sources/Uses

Two questions before anything else: is the price reasonable relative to what the
target is actually worth on a standalone basis, and does the proposed financing
plan actually add up to the purchase price?

In [ ]:
def premium_analysis(target: CompanyProfile, deal_terms: DealTerms) -> dict:
    """Compare the agreed purchase price to the target's standalone value."""
    standalone_value = (
        target.standalone_value
        if target.standalone_value is not None
        else target.ebitda * target.fallback_ev_ebitda_multiple
    )
    premium_pct = (
        ((deal_terms.purchase_price - standalone_value) / standalone_value) * 100
        if standalone_value else 0
    )
    return {
        "target_standalone_value": round(standalone_value, 2),
        "purchase_price": deal_terms.purchase_price,
        "premium_pct": round(premium_pct, 1),
    }


def sources_and_uses(deal_terms: DealTerms, financing: FinancingAssumptions, purchase_price: float) -> dict:
    """Check that the financing plan actually funds the deal. Advisory/transaction
    fees are excluded for simplicity — add a fees line if you need precision here."""
    cash_needed = purchase_price * deal_terms.cash_component_pct
    stock_needed = purchase_price * deal_terms.stock_component_pct
    cash_sources = financing.acquirer_cash_used + financing.new_debt_raised
    cash_funding_gap = round(cash_needed - cash_sources, 2)
    # stock_needed is in Rs Cr; acquirer_share_price is in plain Rs — convert before dividing
    new_shares_issued = (
        (stock_needed * 1e7) / deal_terms.acquirer_share_price
        if deal_terms.acquirer_share_price else 0
    )

    return {
        "cash_needed": round(cash_needed, 2),
        "stock_needed": round(stock_needed, 2),
        "cash_sources": round(cash_sources, 2),
        "cash_funding_gap": cash_funding_gap,
        "new_shares_issued": round(new_shares_issued, 0),
    }

## 4. Combined-Entity Impact — Pro-Forma, Synergies, Accretion/Dilution, Leverage

This is the core of the feasibility question: what does the combined entity actually
look like, does the deal help or hurt the acquirer's shareholders (accretion/dilution),
do the synergy assumptions justify the price, and can the combined entity carry the
resulting debt load.

Simplifying assumption, stated explicitly: the target's existing net debt is assumed
onto the combined balance sheet (purchase price is treated as equity value paid for
the target, consistent with how `standalone_value` is computed in Pipeline 1).

In [ ]:
def pro_forma_combined(acquirer: CompanyProfile, target: CompanyProfile, financing: FinancingAssumptions) -> dict:
    """Combine the two standalone P&Ls and layer on the interest cost of new debt.
    Pre-synergy — this is the day-one picture before any operational integration."""
    combined_revenue = acquirer.revenue + target.revenue
    combined_ebitda_pre_synergy = acquirer.ebitda + target.ebitda

    additional_interest_pretax = financing.new_debt_raised * financing.cost_of_new_debt
    additional_interest_after_tax = additional_interest_pretax * (1 - financing.tax_rate)

    combined_net_income_pre_synergy = (
        acquirer.net_income + target.net_income - additional_interest_after_tax
    )

    return {
        "combined_revenue": round(combined_revenue, 2),
        "combined_ebitda_pre_synergy": round(combined_ebitda_pre_synergy, 2),
        "additional_interest_after_tax": round(additional_interest_after_tax, 2),
        "combined_net_income_pre_synergy": round(combined_net_income_pre_synergy, 2),
    }


def synergy_npv(synergies: SynergyAssumptions, tax_rate: float) -> dict:
    """NPV of synergies: linear ramp to full run-rate over `ramp_up_years`, then a
    flat (no-growth) perpetuity from full run-rate — deliberately conservative,
    doesn't assume synergies keep growing forever."""
    full_run_rate_pretax = (
        synergies.annual_cost_synergies
        + synergies.annual_revenue_synergies * synergies.synergy_ebitda_margin
    )
    full_run_rate_after_tax = full_run_rate_pretax * (1 - tax_rate)

    ramp_cashflows = [
        full_run_rate_after_tax * (yr / synergies.ramp_up_years)
        for yr in range(1, synergies.ramp_up_years + 1)
    ]
    pv_ramp = sum(
        cf / ((1 + synergies.synergy_discount_rate) ** t)
        for t, cf in enumerate(ramp_cashflows, start=1)
    )

    perpetuity_value = full_run_rate_after_tax / synergies.synergy_discount_rate
    pv_perpetuity = perpetuity_value / ((1 + synergies.synergy_discount_rate) ** synergies.ramp_up_years)

    total_synergy_npv = pv_ramp + pv_perpetuity

    return {
        "full_run_rate_pretax": round(full_run_rate_pretax, 2),
        "full_run_rate_after_tax": round(full_run_rate_after_tax, 2),
        "pv_ramp_period": round(pv_ramp, 2),
        "pv_terminal_perpetuity": round(pv_perpetuity, 2),
        "total_synergy_npv": round(total_synergy_npv, 2),
    }

In [ ]:
def accretion_dilution(acquirer: CompanyProfile, pro_forma: dict, su: dict, synergy: dict) -> dict:
    """Does the deal help or hurt the acquirer's existing shareholders, on an EPS
    basis? Shown both pre-synergy (day one) and at full synergy run-rate."""
    acquirer_standalone_eps = (acquirer.net_income * 1e7) / acquirer.shares_outstanding
    pro_forma_shares = acquirer.shares_outstanding + su["new_shares_issued"]

    pro_forma_eps_pre_synergy = (pro_forma["combined_net_income_pre_synergy"] * 1e7) / pro_forma_shares
    change_pre_synergy_pct = (pro_forma_eps_pre_synergy - acquirer_standalone_eps) / acquirer_standalone_eps

    net_income_post_synergy = pro_forma["combined_net_income_pre_synergy"] + synergy["full_run_rate_after_tax"]
    pro_forma_eps_post_synergy = (net_income_post_synergy * 1e7) / pro_forma_shares
    change_post_synergy_pct = (pro_forma_eps_post_synergy - acquirer_standalone_eps) / acquirer_standalone_eps

    return {
        "acquirer_standalone_eps": round(acquirer_standalone_eps, 2),
        "pro_forma_shares": round(pro_forma_shares, 0),
        "pro_forma_eps_pre_synergy": round(pro_forma_eps_pre_synergy, 2),
        "change_pre_synergy_pct": round(change_pre_synergy_pct * 100, 1),
        "pro_forma_eps_post_synergy_full_run_rate": round(pro_forma_eps_post_synergy, 2),
        "change_post_synergy_pct": round(change_post_synergy_pct * 100, 1),
    }


def leverage_feasibility(acquirer: CompanyProfile, target: CompanyProfile,
                          financing: FinancingAssumptions, pro_forma: dict,
                          threshold: float = 4.0) -> dict:
    """Pro-forma Net Debt / EBITDA against a feasibility threshold. 4.0x is a
    reasonable default ceiling for SME lending covenants — adjust per the CA's
    actual banking relationships if they have a tighter or looser covenant."""
    pro_forma_net_debt = (
        acquirer.net_debt + target.net_debt
        + financing.new_debt_raised + financing.acquirer_cash_used
    )
    combined_ebitda = pro_forma["combined_ebitda_pre_synergy"]
    leverage_ratio = pro_forma_net_debt / combined_ebitda if combined_ebitda else float("inf")

    return {
        "pro_forma_net_debt": round(pro_forma_net_debt, 2),
        "leverage_ratio": round(leverage_ratio, 2),
        "leverage_threshold": threshold,
        "leverage_flagged": leverage_ratio > threshold,
    }


def breakeven_synergy_required(accretion: dict, pro_forma: dict, financing: FinancingAssumptions) -> dict:
    """If the deal is dilutive pre-synergy, how much after-tax synergy is needed
    just to get back to EPS-neutral? Zero if the deal is already accretive."""
    if accretion["change_pre_synergy_pct"] >= 0:
        return {
            "breakeven_after_tax_synergy_required": 0.0,
            "breakeven_pretax_synergy_required": 0.0,
            "note": "Deal is already EPS-neutral or accretive before synergies.",
        }

    required_net_income = accretion["acquirer_standalone_eps"] * accretion["pro_forma_shares"] / 1e7
    current_net_income = pro_forma["combined_net_income_pre_synergy"]
    required_after_tax = required_net_income - current_net_income
    required_pretax = required_after_tax / (1 - financing.tax_rate)

    return {
        "breakeven_after_tax_synergy_required": round(required_after_tax, 2),
        "breakeven_pretax_synergy_required": round(required_pretax, 2),
    }


def overall_feasibility_verdict(premium: dict, leverage: dict, accretion: dict,
                                 synergy: dict, breakeven: dict) -> dict:
    """Rule-based verdict, not a judgment call — every reason traces to a specific
    flag computed above. Thresholds (50% premium, -10% dilution) are defaults;
    tune them per deal type or the CA's own risk appetite."""
    reasons = []

    if leverage["leverage_flagged"]:
        reasons.append(
            f"Pro-forma leverage of {leverage['leverage_ratio']}x exceeds the "
            f"{leverage['leverage_threshold']}x threshold."
        )
    if premium["premium_pct"] > 50:
        reasons.append(
            f"Purchase price implies a {premium['premium_pct']}% premium to standalone "
            f"value — unusually high, re-examine assumptions."
        )
    if (accretion["change_pre_synergy_pct"] < -10
            and synergy["total_synergy_npv"] < breakeven.get("breakeven_after_tax_synergy_required", 0) * 3):
        reasons.append(
            "Deal is meaningfully dilutive pre-synergy and synergy NPV does not "
            "comfortably cover the breakeven requirement."
        )

    if not reasons:
        verdict = "FEASIBLE AS STRUCTURED"
    elif len(reasons) == 1:
        verdict = "FEASIBLE WITH CONDITIONS — review flagged item below"
    else:
        verdict = "NOT RECOMMENDED AS STRUCTURED — multiple flags raised"

    return {"verdict": verdict, "reasons": reasons}

## 5. Orchestration — Run the Full Engine

One function that calls every calculation above in the right order and returns a
single result dict — this is what the report generator consumes.

In [ ]:
def calculate_deal_feasibility(req: DealFeasibilityRequest) -> dict:
    """Runs the full deterministic feasibility check end to end."""
    premium = premium_analysis(req.target, req.deal_terms)
    su = sources_and_uses(req.deal_terms, req.financing, premium["purchase_price"])
    pro_forma = pro_forma_combined(req.acquirer, req.target, req.financing)
    synergy = synergy_npv(req.synergies, req.financing.tax_rate)
    accretion = accretion_dilution(req.acquirer, pro_forma, su, synergy)
    leverage = leverage_feasibility(req.acquirer, req.target, req.financing, pro_forma)
    breakeven = breakeven_synergy_required(accretion, pro_forma, req.financing)
    verdict = overall_feasibility_verdict(premium, leverage, accretion, synergy, breakeven)

    return {
        "premium": premium,
        "sources_and_uses": su,
        "pro_forma": pro_forma,
        "synergy": synergy,
        "accretion_dilution": accretion,
        "leverage": leverage,
        "breakeven": breakeven,
        "verdict": verdict,
    }

## 6. Report Generator (Templated, Not Generative)

Same principle as Pipeline 1 — plain string formatting pulling from the calculated
dict, no LLM, every line traceable.

In [ ]:
def build_deal_feasibility_report(req: DealFeasibilityRequest, result: dict) -> str:
    """Templated report generation — plain string formatting, no LLM."""
    lines = []
    lines.append(f"DEAL FEASIBILITY REPORT — {req.meta.acquirer_name} / {req.meta.target_name}")
    lines.append(f"Prepared for: {req.meta.ca_firm_name}")
    lines.append(f"Date: {req.meta.report_date}")
    lines.append(f"Deal type: {req.deal_terms.deal_type}")
    lines.append(f"Rationale: {req.meta.deal_rationale}")
    lines.append(f"Sector: {req.meta.sector}")
    lines.append("=" * 60)

    p = result["premium"]
    lines.append("\n--- PREMIUM ANALYSIS ---")
    lines.append(f"Target standalone value: Rs. {p['target_standalone_value']:.2f} Cr")
    lines.append(f"Agreed purchase price:   Rs. {p['purchase_price']:.2f} Cr")
    lines.append(f"Premium to standalone value: {p['premium_pct']:.1f}%")

    su = result["sources_and_uses"]
    lines.append("\n--- SOURCES & USES ---")
    lines.append(f"Cash required:   Rs. {su['cash_needed']:.2f} Cr")
    lines.append(f"Stock required:  Rs. {su['stock_needed']:.2f} Cr  (new shares issued: {su['new_shares_issued']:.0f})")
    lines.append(f"Cash sources (acquirer cash + new debt): Rs. {su['cash_sources']:.2f} Cr")
    if abs(su["cash_funding_gap"]) > 0.01:
        lines.append(f"FLAG: Funding gap of Rs. {su['cash_funding_gap']:.2f} Cr — sources do not match cash required.")
    else:
        lines.append("Sources and uses balance.")

    pf = result["pro_forma"]
    lines.append("\n--- PRO-FORMA COMBINED ENTITY (pre-synergy) ---")
    lines.append(f"Combined Revenue: Rs. {pf['combined_revenue']:.2f} Cr")
    lines.append(f"Combined EBITDA:  Rs. {pf['combined_ebitda_pre_synergy']:.2f} Cr")
    lines.append(f"Combined Net Income (after new-debt interest): Rs. {pf['combined_net_income_pre_synergy']:.2f} Cr")

    syn = result["synergy"]
    lines.append("\n--- SYNERGY NPV ---")
    lines.append(f"Full run-rate synergies (after-tax): Rs. {syn['full_run_rate_after_tax']:.2f} Cr/year")
    lines.append(f"PV during ramp-up period:  Rs. {syn['pv_ramp_period']:.2f} Cr")
    lines.append(f"PV of terminal perpetuity: Rs. {syn['pv_terminal_perpetuity']:.2f} Cr")
    lines.append(f"Total Synergy NPV:         Rs. {syn['total_synergy_npv']:.2f} Cr")

    ad = result["accretion_dilution"]
    lines.append("\n--- ACCRETION / DILUTION ---")
    lines.append(f"Acquirer standalone EPS: Rs. {ad['acquirer_standalone_eps']:.2f}")
    lines.append(f"Pro-forma EPS (pre-synergy):            Rs. {ad['pro_forma_eps_pre_synergy']:.2f}  ({ad['change_pre_synergy_pct']:+.1f}%)")
    lines.append(f"Pro-forma EPS (full run-rate synergy):  Rs. {ad['pro_forma_eps_post_synergy_full_run_rate']:.2f}  ({ad['change_post_synergy_pct']:+.1f}%)")

    lev = result["leverage"]
    lines.append("\n--- LEVERAGE & FINANCING FEASIBILITY ---")
    lines.append(f"Pro-forma Net Debt: Rs. {lev['pro_forma_net_debt']:.2f} Cr")
    lines.append(f"Pro-forma Net Debt / EBITDA: {lev['leverage_ratio']:.2f}x  (threshold: {lev['leverage_threshold']:.1f}x)")
    if lev["leverage_flagged"]:
        lines.append("FLAG: Pro-forma leverage exceeds the feasibility threshold.")
    else:
        lines.append("Pro-forma leverage is within the feasibility threshold.")

    be = result["breakeven"]
    lines.append("\n--- BREAKEVEN SYNERGY REQUIRED ---")
    if be.get("breakeven_after_tax_synergy_required", 0) > 0:
        lines.append(f"After-tax synergy needed for EPS-neutrality: Rs. {be['breakeven_after_tax_synergy_required']:.2f} Cr/year")
        lines.append(f"Pre-tax equivalent: Rs. {be['breakeven_pretax_synergy_required']:.2f} Cr/year")
    else:
        lines.append(be.get("note", "Deal is already EPS-neutral or accretive before synergies."))

    v = result["verdict"]
    lines.append("\n--- OVERALL FEASIBILITY VERDICT ---")
    lines.append(v["verdict"])
    for r in v["reasons"]:
        lines.append(f"  - {r}")

    lines.append("\n" + "=" * 60)
    lines.append("This report is formula-driven and fully auditable. Every figure above")
    lines.append("traces to the assumptions and inputs supplied for this engagement.")
    lines.append("Prepared via Vaelo — for review by the submitting CA before client delivery.")

    return "\n".join(lines)

## 7. End-to-End Example Run

Illustrative figures only — replace with a real acquirer/target/deal before using
this for an actual engagement. Worth also running once with deliberately aggressive
inputs (high leverage, high premium) to confirm the flags actually trigger, the same
sanity-check discipline used in Pipeline 1.

In [ ]:
# --- Example DealFeasibilityRequest (replace with real deal data) ---

example_deal = DealFeasibilityRequest(
    meta=DealMeta(
        acquirer_name="Acquirer Pvt Ltd",
        target_name="Target Pvt Ltd",
        ca_firm_name="Example & Associates",
        deal_rationale="horizontal acquisition — market expansion",
        sector="Manufacturing",
    ),
    acquirer=CompanyProfile(
        name="Acquirer Pvt Ltd",
        revenue=45.0,
        ebitda=7.5,
        net_income=4.2,
        shares_outstanding=5_000_000,
        net_debt=6.0,
    ),
    target=CompanyProfile(
        name="Target Pvt Ltd",
        revenue=15.0,
        ebitda=2.4,
        net_income=1.3,
        shares_outstanding=1_000_000,
        net_debt=2.0,
        standalone_value=None,               # plug in a Pipeline 1 DCF value here if available
        fallback_ev_ebitda_multiple=6.5,
    ),
    deal_terms=DealTerms(
        deal_type="acquisition",
        purchase_price=16.5,                 # Rs Cr, agreed equity value for target
        cash_component_pct=0.6,
        stock_component_pct=0.4,
        acquirer_share_price=850,            # Rs
    ),
    financing=FinancingAssumptions(
        new_debt_raised=6.0,
        cost_of_new_debt=0.115,
        acquirer_cash_used=3.9,
        tax_rate=0.25,
    ),
    synergies=SynergyAssumptions(
        annual_cost_synergies=0.8,
        annual_revenue_synergies=2.0,
        synergy_ebitda_margin=0.15,
        ramp_up_years=3,
        synergy_discount_rate=0.14,
    ),
)

# --- Run the pipeline ---

result = calculate_deal_feasibility(example_deal)
report = build_deal_feasibility_report(example_deal, result)
print(report)

## 8. Stress Test — Confirming the Flags Actually Fire

A clean example run (Section 7) only proves the happy path — it can't tell you
whether the verdict logic's flags (leverage threshold, premium threshold,
dilution+synergy combined check) actually trigger when they should. Untested
conditional logic is exactly where bugs hide silently, so this run deliberately
uses an aggressive, overpriced, over-levered deal to confirm each flag fires
correctly, and that flags which shouldn't fire (e.g. leverage here, since it's
pushed close to but still under the threshold) correctly don't.

In [ ]:
# --- Stress test: aggressive/dilutive deal (deliberately unfavorable inputs) ---

stress_deal = DealFeasibilityRequest(
    meta=DealMeta(
        acquirer_name="Acquirer Pvt Ltd",
        target_name="Overpriced Target Pvt Ltd",
        ca_firm_name="Example & Associates",
        deal_rationale="stress test — high premium, high leverage, low synergy",
        sector="Manufacturing",
    ),
    acquirer=CompanyProfile(
        name="Acquirer Pvt Ltd", revenue=45.0, ebitda=7.5, net_income=4.2,
        shares_outstanding=5_000_000, net_debt=6.0,
    ),
    target=CompanyProfile(
        name="Overpriced Target Pvt Ltd", revenue=15.0, ebitda=2.4, net_income=1.3,
        shares_outstanding=1_000_000, net_debt=2.0,
        standalone_value=None, fallback_ev_ebitda_multiple=6.5,
    ),
    deal_terms=DealTerms(
        deal_type="acquisition",
        purchase_price=30.0,   # well above the ~15.6 Cr standalone value -> high premium
        cash_component_pct=0.9,
        stock_component_pct=0.1,
        acquirer_share_price=850,
    ),
    financing=FinancingAssumptions(
        new_debt_raised=24.0,   # heavy new debt -> pushes leverage up, but should stay under 4.0x here
        cost_of_new_debt=0.13,
        acquirer_cash_used=3.0,
        tax_rate=0.25,
    ),
    synergies=SynergyAssumptions(
        annual_cost_synergies=0.2,
        annual_revenue_synergies=0.3,
        synergy_ebitda_margin=0.10,
        ramp_up_years=3,
        synergy_discount_rate=0.14,
    ),
)

stress_result = calculate_deal_feasibility(stress_deal)
stress_report = build_deal_feasibility_report(stress_deal, stress_result)
print(stress_report)

In [ ]:
# --- Confirm the flags fired (or correctly did not fire) as expected ---

assert stress_result["premium"]["premium_pct"] > 50, "Expected premium flag to trigger"
assert not stress_result["leverage"]["leverage_flagged"], (
    "Expected leverage to stay under threshold in this scenario — "
    "if this fails, either the scenario or the threshold logic needs review"
)
assert stress_result["accretion_dilution"]["change_pre_synergy_pct"] < -10, (
    "Expected meaningful pre-synergy dilution in this scenario"
)
assert len(stress_result["verdict"]["reasons"]) >= 2, (
    "Expected multiple flags to be raised for this deliberately unfavorable deal"
)
assert stress_result["verdict"]["verdict"] == "NOT RECOMMENDED AS STRUCTURED — multiple flags raised"

print("All flag-logic assertions passed — verdict escalation confirmed working in both directions.")